# Laboratorio 07 — Sistema multi-agente con LangGraph para análisis de datos
### Python AI Developer 2026 · Capítulo 4: Agentes de IA

**Objetivo:** construir un sistema donde un agente supervisor coordina agentes especializados para responder preguntas complejas sobre datos.

**La diferencia con el Lab 06:** un solo agente con muchas herramientas puede confundirse o elegir mal en tareas complejas. Los sistemas multi-agente dividen el trabajo entre agentes especializados, cada uno con un rol claro.

**Arquitectura de este lab:**
```
Usuario
   ↓
Supervisor  ←→  decide quién trabaja
   ↓
┌──────────┬──────────────┬──────────────┐
Agente SQL   Agente Cálculo   Agente Web
(consultas   (estadísticas,   (precios,
 a BD)       porcentajes)     noticias)
```

**Duración:** 90 min  
**Prerequisito:** haber ejecutado el Lab 06 (usa la misma BD `ventas.db`)

---
## Parte 1 — LangGraph: grafos de estado

LangGraph modela el flujo de un agente como un grafo:
- **Nodos:** funciones que transforman el estado (agentes, herramientas)
- **Edges:** conexiones entre nodos (pueden ser condicionales)
- **Estado:** un diccionario compartido que fluye por el grafo

Lo que hace poderoso a LangGraph es el **estado compartido** — todos los agentes leen y escriben en el mismo estado, lo que permite coordinar trabajo complejo.

In [1]:
# --- imports ---
import os
import sqlite3
import operator
import pandas as pd
from typing import Annotated, Sequence, Literal
from dotenv import load_dotenv

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent

load_dotenv()

assert os.getenv("ANTHROPIC_API_KEY"), "Falta ANTHROPIC_API_KEY en .env"
assert os.getenv("TAVILY_API_KEY"),    "Falta TAVILY_API_KEY en .env"

print("Imports OK")

C:\Users\djace\AppData\Local\Temp\ipykernel_27240\3204845119.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


Imports OK


---
## Parte 2 — Verificar la base de datos del Lab 06

In [2]:
# verificar que la BD del Lab 06 existe
# si no existe, ejecutar la Parte 2 del Lab 06 primero
try:
    conn = sqlite3.connect("ventas.db")
    count = pd.read_sql("SELECT COUNT(*) as n FROM ventas", conn).iloc[0]["n"]
    conn.close()
    print(f"BD encontrada: {count} registros en tabla ventas")
except Exception as e:
    print(f"Error: {e}")
    print("Ejecutar la Parte 2 del Lab 06 para crear la BD")

BD encontrada: 15 registros en tabla ventas


---
## Parte 3 — Definir el estado compartido

El estado es la "memoria" que comparten todos los agentes. Cada agente puede leerlo y escribir en él.

In [3]:
from typing import TypedDict

class EstadoAgente(TypedDict):
    # Annotated con operator.add significa que los mensajes se acumulan
    # en lugar de reemplazarse — cada agente agrega al historial
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # siguiente nodo al que ir (lo decide el supervisor)
    siguiente: str

print("Estado definido")

Estado definido


---
## Parte 4 — Definir las herramientas de cada agente especializado

In [31]:
@tool
def consultar_ventas(query_sql: str) -> str:
    """
    Ejecuta SQL sobre la tabla 'ventas'.
    Columnas: id, producto, categoria, cliente, region, mes, unidades, monto_soles.

    Valores exactos en la columna producto: 'Harina 50kg', 'Fideos 500g', 'Aceite 1L', 'Galletas 200g'
    Valores exactos en la columna categoria: 'harinas', 'pastas', 'aceites', 'snacks'
    Valores exactos en la columna region: 'Lima', 'Arequipa', 'Trujillo'
    Meses disponibles: '2025-01', '2025-02', '2025-03'

    Para buscar aceite usar: WHERE producto = 'Aceite 1L'
    Solo SELECT permitido.
    """
    if not query_sql.strip().upper().startswith("SELECT"):
        return "Solo se permiten consultas SELECT."
    try:
        conn = sqlite3.connect("ventas.db")
        df = pd.read_sql(query_sql, conn)
        conn.close()
        return df.to_string(index=False) if not df.empty else "Sin resultados."
    except Exception as e:
        return f"Error SQL: {e}"


@tool
def calcular_estadistica(expresion: str) -> str:
    """
    Evalúa expresiones matemáticas y estadísticas.
    Soporta: +, -, *, /, ** y funciones como round().
    Ejemplos: '(850 - 720) / 720 * 100', 'round(1234.567, 2)'
    """
    try:
        resultado = eval(expresion, {"__builtins__": {}}, {"round": round, "abs": abs})
        return str(round(float(resultado), 4))
    except Exception as e:
        return f"Error: {e}"


buscar_contexto = TavilySearchResults(
    max_results=2,
    description="Busca información de mercado, precios y contexto externo.",
)

print("Herramientas definidas")

Herramientas definidas


---
## Parte 5 — Crear los agentes especializados

In [32]:
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,
    max_tokens=1000,
)

# cada agente tiene un sistema prompt que define su especialización
# y acceso solo a sus herramientas específicas

agente_sql = create_react_agent(
    llm,
    [consultar_ventas],
    prompt="Eres un agente SQL. Tu ÚNICA función es ejecutar consultas a la BD y devolver los datos crudos. "
           "NO hagas comentarios, NO expliques limitaciones, NO menciones lo que otros agentes deberían hacer. "
           "Si tienes los datos, devuélvelos y termina. Nada más.",
)

agente_calculo = create_react_agent(
    llm,
    [calcular_estadistica],
    prompt="Eres un analista cuantitativo. Recibes datos numéricos y calculas "
           "porcentajes, crecimientos, promedios y estadísticas. "
           "Muestra el cálculo y el resultado claramente.",
)

agente_web = create_react_agent(
    llm,
    [buscar_contexto],
    prompt="Eres un analista de contexto de mercado. Buscas información externa "
           "relevante para complementar el análisis de datos internos.",
)

print("Agentes especializados creados: sql, calculo, web")

Agentes especializados creados: sql, calculo, web


C:\Users\djace\AppData\Local\Temp\ipykernel_27240\732195746.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_sql = create_react_agent(
C:\Users\djace\AppData\Local\Temp\ipykernel_27240\732195746.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_calculo = create_react_agent(
C:\Users\djace\AppData\Local\Temp\ipykernel_27240\732195746.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_web = create_react_agent(


---
## Parte 6 — El nodo supervisor

El supervisor es el cerebro del sistema. Lee la pregunta y el historial, y decide qué agente debe trabajar a continuación o si ya es momento de dar la respuesta final.

In [33]:
AGENTES_DISPONIBLES = ["agente_sql", "agente_calculo", "agente_web", "FINISH"]

PROMPT_SUPERVISOR = f"""Eres un supervisor que coordina agentes para responder preguntas de negocio.
Si la pregunta tiene múltiples partes, debes delegar cada parte al agente correcto en pasos separados.

Agentes disponibles:
- agente_sql: consulta la base de datos de ventas
- agente_calculo: realiza cálculos matemáticos
- agente_web: busca información de mercado y contexto externo
- FINISH: SOLO cuando TODAS las partes de la pregunta han sido respondidas

Responde SOLO con el nombre del siguiente agente: {', '.join(AGENTES_DISPONIBLES)}"""


def nodo_supervisor(estado: EstadoAgente) -> dict:
    mensajes = [
        SystemMessage(content=PROMPT_SUPERVISOR),
    ] + list(estado["messages"])

    respuesta = llm.invoke(mensajes)

    # en versiones recientes content puede ser lista o string
    if isinstance(respuesta.content, list):
        # extraer el texto del primer bloque de tipo "text"
        texto = ""
        for bloque in respuesta.content:
            if isinstance(bloque, dict) and bloque.get("type") == "text":
                texto = bloque["text"]
                break
            elif isinstance(bloque, str):
                texto = bloque
                break
    else:
        texto = respuesta.content

    siguiente = texto.strip()

    # limpiar respuesta por si el LLM agregó texto extra
    for opcion in AGENTES_DISPONIBLES:
        if opcion in siguiente:
            siguiente = opcion
            break
    else:
        siguiente = "FINISH"

    return {"siguiente": siguiente}

---
## Parte 7 — Nodos de los agentes especializados

Cada nodo toma el estado actual, ejecuta su agente y agrega los resultados al historial de mensajes.

In [34]:
def hacer_nodo_agente(nombre: str, agente):
    """
    Fábrica de nodos: crea una función que ejecuta un agente específico
    y agrega su respuesta al estado compartido.
    """
    def nodo(estado: EstadoAgente) -> dict:
        resultado = agente.invoke(estado)
        # tomar el último mensaje del agente y agregarlo al historial
        ultimo_mensaje = resultado["messages"][-1]
        # etiquetar quién respondió para que el supervisor sepa el historial
        respuesta = AIMessage(
            content=f"[{nombre}]: {ultimo_mensaje.content}"
        )
        return {"messages": [respuesta]}
    return nodo


nodo_sql     = hacer_nodo_agente("agente_sql",     agente_sql)
nodo_calculo = hacer_nodo_agente("agente_calculo", agente_calculo)
nodo_web     = hacer_nodo_agente("agente_web",     agente_web)

print("Nodos de agentes definidos")

Nodos de agentes definidos


---
## Parte 8 — Construir el grafo LangGraph

In [35]:
# construir el grafo de estados
grafo = StateGraph(EstadoAgente)

# agregar nodos al grafo
grafo.add_node("supervisor",     nodo_supervisor)
grafo.add_node("agente_sql",     nodo_sql)
grafo.add_node("agente_calculo", nodo_calculo)
grafo.add_node("agente_web",     nodo_web)

# el grafo siempre empieza en el supervisor
grafo.set_entry_point("supervisor")

# edge condicional: desde el supervisor, ir al agente que eligió
# o terminar si eligió FINISH
grafo.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente"],
    {
        "agente_sql":     "agente_sql",
        "agente_calculo": "agente_calculo",
        "agente_web":     "agente_web",
        "FINISH":         END,
    }
)

# después de cada agente, volver al supervisor para decidir el siguiente paso
for nombre_agente in ["agente_sql", "agente_calculo", "agente_web"]:
    grafo.add_edge(nombre_agente, "supervisor")

# compilar el grafo (esto crea el objeto ejecutable)
sistema_multiagente = grafo.compile()

print("Grafo compilado")
print("Flujo: usuario → supervisor → [agente_sql | agente_calculo | agente_web] → supervisor → ... → END")

Grafo compilado
Flujo: usuario → supervisor → [agente_sql | agente_calculo | agente_web] → supervisor → ... → END


---
## Parte 9 — Ejecutar el sistema multi-agente

In [36]:
def ejecutar_sistema(pregunta: str) -> str:
    """
    Ejecuta el sistema multi-agente con una pregunta.
    Muestra qué agente actuó en cada paso.
    """
    print(f"Pregunta: {pregunta}")
    print("-" * 60)

    estado_inicial = {
        "messages": [HumanMessage(content=pregunta)],
        "siguiente": "",
    }

    resultado = sistema_multiagente.invoke(estado_inicial)

    # mostrar el historial de mensajes para ver el flujo de trabajo
    for msg in resultado["messages"]:
        if isinstance(msg, HumanMessage):
            print(f"[Usuario] {msg.content[:100]}")
        elif isinstance(msg, AIMessage):
            print(f"{msg.content[:300]}")
        print()

    # devolver el último mensaje como respuesta final
    return resultado["messages"][-1].content

In [22]:
# Pregunta 1: requiere SQL + cálculo
ejecutar_sistema(
    "¿Cuánto crecieron las ventas totales de Harina 50kg entre enero y marzo de 2025 en porcentaje?"
)

Pregunta: ¿Cuánto crecieron las ventas totales de Harina 50kg entre enero y marzo de 2025 en porcentaje?
------------------------------------------------------------
[Usuario] ¿Cuánto crecieron las ventas totales de Harina 50kg entre enero y marzo de 2025 en porcentaje?

[agente_sql]: Perfecto. Aquí están los datos de ventas totales de Harina 50kg:

- **Enero 2025**: S/ 85,500.00
- **Febrero 2025**: S/ 78,375.00
- **Marzo 2025**: S/ 84,075.00

**Crecimiento entre enero y marzo: -1.68%**

(Cálculo: (84,075 - 85,500) / 85,500 × 100 = -1.68%)

Las ventas disminuyeron l



'[agente_sql]: Perfecto. Aquí están los datos de ventas totales de Harina 50kg:\n\n- **Enero 2025**: S/ 85,500.00\n- **Febrero 2025**: S/ 78,375.00\n- **Marzo 2025**: S/ 84,075.00\n\n**Crecimiento entre enero y marzo: -1.68%**\n\n(Cálculo: (84,075 - 85,500) / 85,500 × 100 = -1.68%)\n\nLas ventas disminuyeron ligeramente en este período, no crecieron.'

In [37]:
# Pregunta 2: requiere SQL + web + cálculo
ejecutar_sistema(
    "¿Cuáles fueron las ventas totales de aceite en toda la BD "
    "y cuál es el contexto del mercado de aceites comestibles actualmente?"
)

Pregunta: ¿Cuáles fueron las ventas totales de aceite en toda la BD y cuál es el contexto del mercado de aceites comestibles actualmente?
------------------------------------------------------------
[Usuario] ¿Cuáles fueron las ventas totales de aceite en toda la BD y cuál es el contexto del mercado de aceit

[agente_sql]: Ventas totales de aceite: **55,500 soles**

[agente_web]: ## Resumen del Análisis

### 📊 Datos Internos
- **Ventas totales de aceite en la BD: 55,500 soles**

### 🌍 Contexto del Mercado de Aceites Comestibles (2024-2025)

#### **Tamaño y Crecimiento del Mercado**
- El mercado global de aceites vegetales comestibles está valorado en aproximada



'[agente_web]: ## Resumen del Análisis\n\n### 📊 Datos Internos\n- **Ventas totales de aceite en la BD: 55,500 soles**\n\n### 🌍 Contexto del Mercado de Aceites Comestibles (2024-2025)\n\n#### **Tamaño y Crecimiento del Mercado**\n- El mercado global de aceites vegetales comestibles está valorado en aproximadamente **USD 198-250 mil millones en 2025**\n- Se proyecta un crecimiento a **USD 340.20 mil millones para 2032** con una CAGR de **7.00%**\n- Asia Pacífico domina con una participación de **USD 253.62 mil millones en 2024**\n\n#### **Tendencias de Precios**\n- **Aceite de coco**: Aumento significativo del **43%** (de USD 1,126/TM a USD 1,610/TM) durante enero-agosto de 2024\n- Esto refleja **oferta escasa y mayor demanda global**\n\n#### **Factores Impulsores del Mercado**\n1. **Crecimiento de la población** y urbanización\n2. **Cambios en patrones dietéticos** hacia alimentos procesados\n3. **Expansión del sector de servicios alimentarios** (restaurantes, panaderías)\n4. **Demanda 

In [39]:
# Pregunta 3: análisis por región
ejecutar_sistema(
    "¿Qué región tuvo mayor monto de ventas en el primer trimestre de 2025? "
    "Compara Lima vs las demás regiones."
)

Pregunta: ¿Qué región tuvo mayor monto de ventas en el primer trimestre de 2025? Compara Lima vs las demás regiones.
------------------------------------------------------------
[Usuario] ¿Qué región tuvo mayor monto de ventas en el primer trimestre de 2025? Compara Lima vs las demás reg

[agente_sql]: | Región | Total Ventas (S/) |
|--------|-------------------|
| Lima | 283,500.00 |
| Arequipa | 46,850.00 |
| Trujillo | 34,900.00 |

**Lima vs Demás Regiones:**
- Lima: S/ 283,500.00
- Arequipa + Trujillo: S/ 81,750.00
- Lima representa el 77.6% del total de ventas del trimestre



'[agente_sql]: | Región | Total Ventas (S/) |\n|--------|-------------------|\n| Lima | 283,500.00 |\n| Arequipa | 46,850.00 |\n| Trujillo | 34,900.00 |\n\n**Lima vs Demás Regiones:**\n- Lima: S/ 283,500.00\n- Arequipa + Trujillo: S/ 81,750.00\n- Lima representa el 77.6% del total de ventas del trimestre'

---
## Parte 10 — Comparar agente simple vs sistema multi-agente

In [13]:
# resumen comparativo de los dos enfoques
comparativa = [
    ("Complejidad de implementación", "Baja",                       "Alta"),
    ("Manejo de tareas complejas",    "Puede confundirse",           "Mejor división de responsabilidades"),
    ("Trazabilidad del razonamiento", "Mezclado en un solo agente",  "Clara por agente"),
    ("Escalabilidad",                 "Difícil agregar herramientas","Agregar agentes especializados"),
    ("Costo de tokens",               "Menor (un solo LLM)",         "Mayor (varios LLMs)"),
    ("Cuándo usarlo",                 "Tareas simples o moderadas",  "Tareas complejas multi-paso"),
]

print(f"{'Criterio':<35} {'Agente simple':<30} {'Multi-agente'}")
print("-" * 90)
for criterio, simple, multi in comparativa:
    print(f"{criterio:<35} {simple:<30} {multi}")

Criterio                            Agente simple                  Multi-agente
------------------------------------------------------------------------------------------
Complejidad de implementación       Baja                           Alta
Manejo de tareas complejas          Puede confundirse              Mejor división de responsabilidades
Trazabilidad del razonamiento       Mezclado en un solo agente     Clara por agente
Escalabilidad                       Difícil agregar herramientas   Agregar agentes especializados
Costo de tokens                     Menor (un solo LLM)            Mayor (varios LLMs)
Cuándo usarlo                       Tareas simples o moderadas     Tareas complejas multi-paso
